# Display of time series data recorded on the Zeiss Cell Discoverer 7
_______________
This notebook is a method to plot "Region" Data exported from a Zen Blue analysis pipeline. \
The expected input is a csv file containing measurements for individual regions, not region population averages/region stats. \
The generated output will be 96 well plate graphs (.png) and the underlying calculated tables (.csv). \
There are several input prompts along the way.

_______________

### *Disclaimer:* 
This notebook has been developed by Marie Held, image analyst in the Centre for Cell Imaging at The University of Liverpool. 
The source code can be found in: *[CD7 downstream analysis and tools](https://github.com/Marien-kaefer/CD7-downstream-analysis-and-tools)*

_______________

# How to use this notebook
## Structure of a notebook
The notebook contains two types of cell:

Text cells provide information and can be modified by douple-clicking the cell. You are currently reading the text cell. You can create a new text by double clicking on the contents of the cell.

Code cells contain code and the code can be modfied by selecting the cell. To execute the cell press [Shift + Enter] or click the Run button in the tools bar of the Notebook interface above. During exection, an asterisk will appear within the brackets.  After execution is done the asterisk will change into a number, i.e. the nth cell that has been executed during this session. The numbers indicate the order in which the cells have been executed which does not have to follow the order of the cells within the notebook. 

Any code output will be displayed below the cell.

## Before getting started
Before running this notebook, please ensure that you have the data to process accessible. Please note that you can currently **only use .csv files**. 

## Onwards...

# 1. Import required packages
The packages pandas, and matplotlib must be installed in the python environment in which the notebook is run. 

In [10]:
### Import necessary libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 2. Select your input .csv file
Executing the next cell will trigger a request for the path and name of your csv results file generated by the Zen Blue analysis module. Paste the full path and name (incl. extension), e.g.: <code> path\\name.csv </code>, into the input field and hit [Enter]. 

In [11]:
file_path = input()
#file_path = "D:\\FERNIG-FoulkesDan\\2025-02-07_analysis-results\\experiment 7-01_processed Region.csv"

 D:\\FERNIG-FoulkesDan\\2025-02-07_analysis-results\\experiment 12-01_processed Region.csv


Read the csv file into memory. Depending on the size of the file, this step might take a couple of seconds. Please ignore any DtypeWarning. \
When the cell has finished, the first five rows of the table will be displayed below. 

In [12]:
### extract directory from file path
directory = os.path.dirname(file_path)
filename = os.path.splitext(os.path.basename(file_path))[0]
output_directory = directory + '\\output_directory'

# Check whether the specified output path exists. If it does not exist, create it. 
isExist = os.path.exists(output_directory)
if not isExist:

   # Create a new directory because it does not exist
   os.makedirs(output_directory)
   print("The output directory has been created!")

### Read the input file
df = pd.read_csv(file_path, skiprows=[1])
df_column_names = list(df.columns.values)
df.head()
#print(df.columns.tolist()) 

C:\Users\zeiss\AppData\Local\Temp\ipykernel_11224\1221265020.py:15: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, skiprows=[1])


,ID::ID!!I,ImageSceneContainerName::Image Scene Container Name,ImageSceneRow::Image Scene Row Index!!I,ImageSceneColumn::Image Scene Column Index!!I,ImageSceneName::Image Scene Name,ClassNameParent::Region Class Name Parent,ClassName::Region Class Name,ImageRelativeTime::Image Relative Time,ImageIndexTime::Image Index Time!!I,RegionsCount::Count!!I,Area::Area!!R,AreaConvex::Area Convex!!R,Convexity::Convexity!!R,Compactness::Compactness!!R,Circularity::Circularity!!R,IntensityMean_Calce::Intensity Mean Value of channel 'Calce'!!R,IntensityMean_EtHD1::Intensity Mean Value of channel 'EtHD1'!!R,Live_IntegratedDensity::Live_IntegratedDensity!!R,Dead_IntegratedDensity::Dead_IntegratedDensity!!R
0,2,A1,1,1,P1,Live_Stats,Live,00:00:00,1,1,219.149216,254.080088,1.063589,0.695920,0.707390,1382.126362,1259.183007,275948.96845152,
1,3,A1,1,1,P1,Live_Stats,Live,00:00:00,1,1,667.951531,726.765474,1.051665,0.589662,0.596877,1894.237312,1698.721229,1134663.4466857,
2,4,A1,1,1,P1,Live_Stats,Live,00:00:00,1,1,278.352926,300.581990,1.047106,0.828828,0.828280,1468.433962,1163.698113,323918.77425168,
3,5,A1,1,1,P1,Live_Stats,Live,00:00:00,1,1,454.054258,488.591435,0.979576,0.869332,0.858745,1592.708728,1365.391167,619961.67276168,
4,6,A1,1,1,P1,Live_Stats,Live,00:00:00,1,1,328.485099,395.867375,0.967789,0.673958,0.721881,1475.069767,1175.408430,386104.15467432,


# 3. Pre-processing of the imported data file
Rename the column headers to the names that were visible when setting up the analysis pipeline. The ZenBlue output has column headers that are not easily human readable, hence the renaming. \
When the cell has finished, the first five rows of the table will be displayed below. 

In [13]:
### Rename column headers and specify column types
def rename_column_headers(df_column_names):
    data_type = []
    new_column_names = []
    i = 0
    while i < len(df_column_names):
        name = df_column_names[i]
        name_subset = (name.split("::", 1)[1])
        search_str = "!!"
        if search_str in df_column_names[i]:
            data_type_temp = (name_subset.split("!!", 1)[1])
            if data_type_temp == "I":
                data_type.append("int") 
            elif  data_type_temp == "R":
                data_type.append("float")
        else: 
            data_type.append("str")
        i += 1
        new_column_name = (name_subset.split("!!", 1)[0])
        new_column_names.append(new_column_name)
        del new_column_name      
    del i
    return new_column_names, data_type

#print(df_column_names)

new_column_names, data_type = rename_column_headers(df_column_names)
#print(new_column_names)
#print(data_type)

#Skip the first row and consider integrating unit into column names
df = df.iloc[2:,]

#Rename column names and set data type of columns
i = 0
while i < len(df_column_names):
    df = df.astype({df_column_names[i]: data_type[i]}, errors='ignore')
    df = df.rename(columns={df_column_names[i]: new_column_names[i]})
    i += 1
del i

df['Live_IntegratedDensity'] = pd.to_numeric(df['Live_IntegratedDensity'], errors='coerce')
df['Dead_IntegratedDensity'] = pd.to_numeric(df['Dead_IntegratedDensity'], errors='coerce')

df.head()

,ID,Image Scene Container Name,Image Scene Row Index,Image Scene Column Index,Image Scene Name,Region Class Name Parent,Region Class Name,Image Relative Time,Image Index Time,Count,Area,Area Convex,Convexity,Compactness,Circularity,Intensity Mean Value of channel 'Calce',Intensity Mean Value of channel 'EtHD1',Live_IntegratedDensity,Dead_IntegratedDensity
2,4,A1,1,1,P1,Live_Stats,Live,00:00:00,1,1,278.352926,300.581990,1.047106,0.828828,0.828280,1468.433962,1163.698113,3.239188e+05,NaN
3,5,A1,1,1,P1,Live_Stats,Live,00:00:00,1,1,454.054258,488.591435,0.979576,0.869332,0.858745,1592.708728,1365.391167,6.199617e+05,NaN
4,6,A1,1,1,P1,Live_Stats,Live,00:00:00,1,1,328.485099,395.867375,0.967789,0.673958,0.721881,1475.069767,1175.408430,3.861042e+05,NaN
5,7,A1,1,1,P1,Live_Stats,Live,00:00:00,1,1,167.107245,180.018565,1.079647,0.798173,0.823617,1108.328571,880.225714,1.470921e+05,NaN
6,8,A1,1,1,P1,Live_Stats,Live,00:00:00,1,1,1027.470833,1170.285756,0.894203,0.801790,0.772087,1623.727695,1317.594331,1.353790e+06,NaN


# 4. Choose how many time points to include in the analysis
The dataframe can be filtered by frame index below. The output will show the range of time indeces present in the input file will be printed for reference. \
Example: To filter the time indeces to include 1-10, enter [10] into the input field popping up below and hit [Enter].

In [14]:
print('The time series frames range from ' + str(df['Image Index Time'].min()) + ' to ' + str(df['Image Index Time'].max()))
print('Type the maximum time index to be included for the analysis below and confirm with [Enter]: ')
max_time_index = int(input())

The time series frames range from 1 to 17
Type the maximum time index to be included for the analysis below and confirm with [Enter]: 


 10


In [16]:
df = df[df['Image Index Time'] <= max_time_index] 

# 5. Generate plot grid labels that equate the plate layout
The Layout of the final plot grid equating to a 96 well plate is being created and the plots are generated.


In [17]:
# Cell 6: Create well names
well_name = []
well_rows = ["A", "B", "C", "D", "E", "F", "G", "H"]

x = 0

for n in range(len(well_rows)):  # well rows
    for m in range(12):  # well columns
        well_name.append(well_rows[n] + str(m+1))
        x += 1

# 6. "Plot the Lot"

All the data will be calculated, plotted and the resulting plots and tables saved automatically. This will take time.... 


In [18]:
### pivot tables to calculate population means per time point

live_df = df[df['Region Class Name'] == "Live"]  #live_df.drop(['Region Class Name', 'Region Class Name Parent'], axis=1)
dead_df = df[df['Region Class Name'] == "Dead"]  #dead_df.drop(['Region Class Name', 'Region Class Name Parent'], axis=1)

x_axis_parameter = 'Image Index Time'

for k in range(10, 19):
    parameter_name = new_column_names[k]
    print("Parameter being processed: " + parameter_name)

    
    live_pivot = np.round(pd.pivot_table(live_df, values=parameter_name, 
                                    index=[x_axis_parameter], 
                                    columns='Image Scene Container Name ', 
                                    aggfunc='mean'), 3)
    dead_pivot = np.round(pd.pivot_table(dead_df, values=parameter_name, 
                                    index=[x_axis_parameter], 
                                    columns='Image Scene Container Name ', 
                                    aggfunc='mean'), 3)

    # Live Cell Parameter Plotting
    print('Live Cell Parameter Plotting takes time. Please be patient whilst there is still an asterisk in the square bracket on the left, python is working on plotting the measurements.')
    fig_live_parameter, axes_live_parameter = plt.subplots(nrows=8, ncols=12, sharex=True, sharey=True, figsize=(24, 14))
    x = 0
    
    for n in range(len(well_rows)):  # well rows
        for m in range(12):  # well columns
            #well_name.append(well_rows[n] + str(m+1))
            #print(well_name[x])
            axes_live_parameter[n, m].set(title=well_name[x])
            try:
                #print("Plotting data for well: ", well_name[x]) 
                live_pivot[well_name[x]].plot(x=x_axis_parameter, y=well_name[x], ax=axes_live_parameter[n, m], legend=None)
            except KeyError:
                print("No data available for well: ", well_name[x])        
            x += 1
    
    fig_live_parameter.suptitle("Live Cell " + parameter_name)
    live_pivot.to_csv(output_directory + "\\" +  filename + "_Live-Cell-" + parameter_name + ".csv", index=True) 

    #save
    strFile = output_directory + "\\" +  filename + "_Live-Cell-" + parameter_name + ".png"
    if os.path.isfile(strFile):
       os.remove(strFile)   # Opt.: os.system("rm "+strFile)
    fig_live_parameter.savefig(strFile)
    del strFile

    # Close the figure to prevent display
    plt.close(fig_live_parameter)
    
    # Dead Cell Parameter Plotting
    print('Dead Cell Parameter Plotting takes time. Please be patient whilst there is still an asterisk in the square bracket on the left, python is working on plotting the measurements.')
    fig_dead_parameter, axes_dead_parameter = plt.subplots(nrows=8, ncols=12, sharex=True, sharey=True, figsize=(24, 14))
    x = 0
    
    for n in range(len(well_rows)):  # well rows
        for m in range(12):  # well columns
            #well_name.append(well_rows[n] + str(m+1))
            #print(well_name[x])
            axes_dead_parameter[n, m].set(title=well_name[x])
            try:
                #print("Plotting data for well: ", well_name[x]) 
                dead_pivot[well_name[x]].plot(x=x_axis_parameter, y=well_name[x], ax=axes_dead_parameter[n, m], legend=None)
            except KeyError:
                print("No data available for well: ", well_name[x])        
            x += 1
    
    fig_dead_parameter.suptitle("Dead Cell " + parameter_name)
    live_pivot.to_csv(output_directory + "\\" +  filename + "_Dead-Cell-" + parameter_name + ".csv", index=True) 

    strFile = output_directory + "\\" +  filename + "_Dead-Cell-" + parameter_name + ".png"
    if os.path.isfile(strFile):
       os.remove(strFile)   # Opt.: os.system("rm "+strFile)
    fig_dead_parameter.savefig(strFile)
    del strFile

    # Close the figure to prevent display
    plt.close(fig_dead_parameter)

    del fig_live_parameter, axes_live_parameter, fig_dead_parameter, axes_dead_parameter
    k += 1

print("\n \n DONE WITH INDIVIDUAL PARAMETER PLOTTING \n \n More plotting following now.")

### Calculate live/dead ratio
live_count = np.round(pd.pivot_table(live_df, values=parameter_name, 
                                      index=[x_axis_parameter],                                      
                                      columns='Image Scene Container Name ', 
                                      aggfunc='count'), 3)
dead_count = np.round(pd.pivot_table(dead_df, values=parameter_name, 
                                      index=[x_axis_parameter], 
                                      columns='Image Scene Container Name ', 
                                      aggfunc='count'), 3)

dead_fraction = dead_count.div(dead_count + live_count)

#dead_fraction.head()

### Dead Fraction Plotting
print('Plotting takes time. Please be patient whilst there is still an asterisk in the square bracket on the left, python is working on plotting the measurements.')
fig_fraction, axes_fraction = plt.subplots(nrows=8, ncols=12, sharex=True, sharey=True, figsize=(24, 14))
x = 0

for n in range(len(well_rows)):  # well rows
    for m in range(12):  # well columns
        axes_fraction[n, m].set_title(well_name[x])
        
        try:
            #print("Plotting data for well: ", well_name[x]) 
            dead_fraction[well_name[x]].plot(x=x_axis_parameter, y=well_name[x], ax=axes_fraction[n, m], legend=None)
            axes_fraction[n, m].set_ylim(0, 1)
        except KeyError:
            print("No data available for well: ", well_name[x])        
        x = x + 1

fig_fraction.suptitle("Dead fraction")
dead_fraction.to_csv(output_directory + "\\" +  filename + "_Dead-Fraction.csv", index=True) 
fig_fraction.savefig(output_directory + "\\" +  filename + "_Dead-Fraction.png")

# Close the figure to prevent display
plt.close(fig_fraction)

del fig_fraction, axes_fraction

### Live/Dead Count Plotting and result saving
print('Plotting takes time. Please be patient whilst there is still an asterisk in the square bracket on the left, python is working on plotting the measurements.')
fig_live_count, axes_live_count = plt.subplots(nrows=8, ncols=12, sharex=True, sharey=True, figsize=(24, 14))
x = 0

for n in range(len(well_rows)):  # well rows
    for m in range(12):  # well columns
        axes_live_count[n, m].set(title=well_name[x])
        try:
            #print("Plotting data for well: ", well_name[x]) 
            live_count[well_name[x]].plot(x=x_axis_parameter, y=well_name[x], ax=axes_live_count[n, m], legend=None)
        except KeyError:
            print("No data available for well: ", well_name[x])        
        x = x + 1

fig_live_count.suptitle("Live Count")
fig_live_count.savefig(output_directory + "\\" +  filename + "_Live-Cell-Count.png")
live_count.to_csv(output_directory + "\\" +  filename + "_Live-Cell-Count.csv", index=True) 

# Close the figure to prevent display
plt.close(fig_live_count)

del fig_live_count, axes_live_count

print('Plotting takes time. Please be patient whilst there is still an asterisk in the square bracket on the left, python is working on plotting the measurements.')
fig_dead_count, axes_dead_count = plt.subplots(nrows=8, ncols=12, sharex=True, sharey=True, figsize=(24, 14))
x = 0

for n in range(len(well_rows)):  # well rows
    for m in range(12):  # well columns
        axes_dead_count[n, m].set(title=well_name[x])
        try:
            #print("Plotting data for well: ", well_name[x]) 
            live_count[well_name[x]].plot(x=x_axis_parameter, y=well_name[x], ax=axes_dead_count[n, m], legend=None)
        except KeyError:
            print("No data available for well: ", well_name[x])        
        x = x + 1

fig_dead_count.suptitle("Dead Count")
fig_dead_count.savefig(output_directory + "\\" +  filename + "_Dead-Cell-Count.png")
live_count.to_csv(output_directory + "\\" +  filename + "_Dead-Cell-Count.csv", index=True) 

# Close the figure to prevent display
plt.close(fig_dead_count)

del fig_dead_count, axes_dead_count

%reset -f

print(" \n \n ------------------------------------------- DONE ------------------------------------------- \n \n Back to the start to process the next file!")

Parameter being processed: Area
Live Cell Parameter Plotting takes time. Please be patient whilst there is still an asterisk in the square bracket on the left, python is working on plotting the measurements.
Dead Cell Parameter Plotting takes time. Please be patient whilst there is still an asterisk in the square bracket on the left, python is working on plotting the measurements.
Parameter being processed: Area Convex
Live Cell Parameter Plotting takes time. Please be patient whilst there is still an asterisk in the square bracket on the left, python is working on plotting the measurements.
Dead Cell Parameter Plotting takes time. Please be patient whilst there is still an asterisk in the square bracket on the left, python is working on plotting the measurements.
Parameter being processed: Convexity
Live Cell Parameter Plotting takes time. Please be patient whilst there is still an asterisk in the square bracket on the left, python is working on plotting the measurements.
Dead Cell Pa

# All Done!  
Check your output directory for results files. 
